📥 Bloque 1 – Importación de librerías esenciales

Este bloque carga todas las dependencias requeridas para la automatización:
- pandas, numpy
- sqlalchemy
- pathlib, datetime, time, shutil
- re, unicodedata

Crea el ecosistema base para procesar el archivo Excel.

In [ ]:
import pandas as pd
import re
import unicodedata
import numpy as np
from datetime import datetime
from sqlalchemy import text
import sys
import importlib
import json
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/servihabit/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path.cwd().resolve().parents[2]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()
CONFIG_DIR    = (PROJECT_ROOT / "config").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:",    CONFIG_DIR,    "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"
try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)
    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))
except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "servihabit", "config_servihabit.json")
try:
    with open(config_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


🖥️ Bloque 2 – Configuración de conexión a SQL Server

Define:
- servidor,
- base de datos,
- tabla y schema destino.

Además construye el engine optimizado con:
- fast_executemany,
- pool_pre_ping,
- future=True.

Garantiza una conexión segura, rápida y robusta.

In [ ]:
# --- Parámetros de conexión ---
server   = data["server_config"]["server"]
database = data["server_config"]["database"]
schema   = data["server_config"]["schema"]
table    = data["tablas_remotas"]["tabla_principal"]

driver = "ODBC Driver 17 for SQL Server"

# ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise"
)


📂 Bloque 3 – Configuración del entorno de archivos

Aquí se define:
- Carpeta de origen,
- extensiones permitidas,
- hojas preferidas,
- hoja fallback.

Permite seleccionar y priorizar correctamente el archivo fuente.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "SERVIHABIT").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx",)  # Extensiones de archivo permitidas

PREFERRED_SHEETS = ["Cesantia", "cesantia", "Recaudacion_ServiHabit"]
FALLBACK_SHEET_INDEX = 0


✨ Bloque 4 – Normalización de texto y encabezados

Incluye funciones que:
- eliminan tildes,
- normalizan nombres de columnas,
- eliminan caracteres especiales,
- estandarizan nombres a formato seguro.

Garantiza consistencia entre archivos heterogéneos.

🧠 Bloque 5 – Generación automática de columnas clave

Funciones que extraen información crítica:
- Diccionario de meses (MESES),
- _extract_yyyymm_from_name para detectar períodos,
- canonicalizar_planes para crear nombres canónicos,
- extraer_yyyymm_de_nombre_archivo para obtener períodos.

Permite generar NOMBRE_ARCHIVO y MES_RECAUDACION cuando no vienen en el archivo.

In [50]:
def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))

MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}


def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Intenta extraer YYYYMM desde el nombre del archivo (sin importar la extensión).
    Cubre:
      - YYYYMM
      - YYYY[-_/ .]?MM
      - MM[-_/ .]?YYYY
      - MES(ES) + YYYY (con tildes/abreviaturas) en cualquier orden
    Lanza ValueError si no encuentra período.
    """
    stem = Path(nombre).stem
    stem_norm = _strip_accents(stem).upper()

    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"

    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        for mes_txt, mm in MESES.items():
            if re.search(rf"(?<![A-Z0-9]){mes_txt}(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")


def canonicalizar_planes(nombre: str) -> str:
    """
    Devuelve un nombre canónico estandarizado.
    """
    try:
        yyyymm = _extract_yyyymm_from_name(nombre)
        return f"{yyyymm} Nomina Cesantia.xlsx"
    except ValueError as e:
        print(
            f"⚠️ No se pudo extraer período desde el nombre '{nombre}'. "
            f"Se usará un nombre genérico. Detalle: {e}"
        )
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{timestamp} Nomina Cesantia.xlsx"


def extraer_yyyymm_de_nombre_archivo(
    df: pd.DataFrame,
    nombre_columna: str = "NOMBRE_ARCHIVO"
) -> pd.Series:

    partes = df[nombre_columna].astype(str).str.extract(
        r"(20\d{2})(0[1-9]|1[0-2])",
        expand=True
    )
    serie = (partes[0].fillna("") + partes[1].fillna("")).replace("", pd.NA)
    return serie


🔍 Bloque 6 – Selección del archivo y hoja válida

Lógica que:
- detecta el archivo más reciente,
- intenta abrirlo,
- si está bloqueado → crea copia temporal,
- selecciona hoja por coincidencia o índice.

Asegura que siempre se lea la hoja correcta.

In [ ]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen  = info["archivo_origen"]
excel_path_used = info["excel_path_used"]
_tmp_copy_path  = info["tmp_copy_path"]
target          = info["target_sheet"]
sheet_names     = info["sheet_names"]

print(f"Archivo: {archivo_origen.name}")
print(f"Hoja seleccionada: {target}")


📊 Bloque 7 – Lectura del Excel y limpieza inicial

Incluye:
- descarga segura del Excel,
- eliminación de copia temporal,
- limpieza de textos,
- homogenización de encabezados.

Deja los datos limpios para procesamiento posterior.

In [52]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

if _tmp_copy_path is not None and _tmp_copy_path.exists():
    try:
        _tmp_copy_path.unlink()
        print(f"🗑️ Copia temporal eliminada: {_tmp_copy_path}")
    except Exception as e:
        print(f"⚠️ No se pudo borrar la copia temporal '{_tmp_copy_path}': {e}")

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "nan": "", "NaN": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]

print("Encabezados normalizados:", list(df_raw.columns))

df_raw.head()

Encabezados normalizados: ['folio', 'operacion', 'svs', 'dueno', 'rut', 'cliente', 'plazo', 'cuota', 'subsidio', 'cuota_neta', 'prima_afecta', 'iva_prima_afecta', 'total_prima', 'afecta_neta', 'iva', 'total', 'poliza', 'plan_tecnico', 'mes_carga', 'mes_recaudacion']


,folio,operacion,svs,dueno,rut,cliente,plazo,cuota,subsidio,cuota_neta,prima_afecta,iva_prima_afecta,total_prima,afecta_neta,iva,total,poliza,plan_tecnico,mes_carga,mes_recaudacion
0,1,13280,79,LEASING URBANO,0026869599-0,WILBERT LORVANDAL .,240,5.368199999999999,0,5.3682,0.1503,0.0286,0.1789,5935,1129,7064,8890458,8554,202509 Nomina Cesantia.xlsx,202509
1,2,13278,84,LEASING URBANO,0016663149-1,CARLOS ANDRES VARGAS GONZALEZ,240,4.5151,0,4.5151,0.1264,0.024,0.1504,4991,948,5939,8890458,8554,202509 Nomina Cesantia.xlsx,202509
2,3,13276,80,LEASING URBANO,0025644727-4,JOSE RAFAEL PARRA MEDINA,240,6.991600000000001,0,6.9916,0.1958,0.0372,0.233,7731,1469,9200,8890458,8554,202509 Nomina Cesantia.xlsx,202509
3,4,13275,73,LEASING URBANO,0019574490-4,LAURA ESTER IBAÑEZ FUENTEALBA,240,4.017,0,4.017,0.1125,0.0214,0.1339,4442,845,5287,8890458,8554,202509 Nomina Cesantia.xlsx,202509
4,5,13273,77,LEASING URBANO,0013865957-7,RUBEN EDUARDO LUCERO DIAZ,240,13.524199999999999,0,13.5242,0.3787,0.072,0.4507,14953,2843,17796,8890458,8554,202509 Nomina Cesantia.xlsx,202509


🔄 Bloque 8 – Mapeo dinámico Origen → Destino

Usa la función pick para:
- identificar columnas equivalentes entre rutas distintas de archivos,
- construir df_can estandarizado.

Soporta archivos con encabezados variables sin romper el proceso.

In [53]:
def pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None]*len(df), index=df.index)


# Origen → Destino
guia                = pick(df_raw, "guia")
operacion           = pick(df_raw, "nro_operacion","n_operacion", "no_operacion", "noperacion", "operacion", "folio", "foliocredito")
svs                 = pick(df_raw, "svs", "s_v_s")
dueno               = pick(df_raw, "dueno", "dueño", "nombre_dueno", "nombre_dueño", "nombredueno", "nombredueño")
rut                 = pick(df_raw, "serie_r_u_t", "rut_afiliado", "rutafiliado", "rut", "rutfiliado")
cliente             = pick(df_raw, "nombre_cliente", "nombrecliente", "cliente", "nombre_afiliado", "nombreafiliado")
plazo               = pick(df_raw, "plazo", "plazo_meses", "plazomeses", "plazo_anios", "plazoanios", "plazo_anos", "plazoanos")
cuota               = pick(df_raw, "cuota", "cuota_mensual", "cuotamensual")
subsidio            = pick(df_raw, "subsidio", "monto_subsidio", "montosubsidio")
cuota_neta          = pick(df_raw, "cuota_neta", "cuotaneta", "cuota_mensual_neta", "cuotamensualneta")
prima_afecta_uf     = pick(df_raw, "prima_afecta_uf", "prima_afecta")
iva_afecta_uf       = pick(df_raw, "iva_afecta_uf", "iva_prima_afecta")
total_prima_uf      = pick(df_raw, "total_prima_uf", "total_prima")
afecta_neta_clp     = pick(df_raw, "afecta_neta_clp", "afectaneta_clp", "afecta_neta", "afectaneta")
iva_clp             = pick(df_raw, "iva_clp", "ivaclp", "iva")
total_prima_clp     = pick(df_raw, "total_prima_clp", "totalprimaclp", "total")
poliza              = pick(df_raw, "poliza", "póliza", "n_poliza", "npoliza")
plan_tecnico        = pick(df_raw, "plan_tecnico", "plantecnico", "plan_tecnicolife", "plantecnicolife")
nombre_archivo      = pick(df_raw, "nombre_archivo", "nombrearchivo", "mes_carga")
mes_recaudacion     = pick(df_raw, "mes_recaudacion", "mesrecaudacion", "mes_de_recaudacion", "mesderecaudacion")


df_can = pd.DataFrame({
    "Guia": guia,
    "Operacion": operacion,
    "SVS": svs,
    "Dueno": dueno,
    "Rut": rut,
    "Cliente": cliente,
    "Plazo": plazo,
    "Cuota": cuota,
    "Subsidio": subsidio,
    "Cuota_Neta": cuota_neta,
    "Prima_Afecta_uf": prima_afecta_uf,
    "Iva_Afecta_uf": iva_afecta_uf,
    "Total_Prima_uf": total_prima_uf,
    "Afecta_Neta_clp": afecta_neta_clp,
    "Iva_clp": iva_clp,
    "Total_Prima_CLP": total_prima_clp,
    "Poliza": poliza,
    "Plan_Tecnico": plan_tecnico,
    "NOMBRE_ARCHIVO": nombre_archivo,
    "MES_RECAUDACION": mes_recaudacion
    
})

df_can.head()

,Guia,Operacion,SVS,Dueno,Rut,Cliente,Plazo,Cuota,Subsidio,Cuota_Neta,Prima_Afecta_uf,Iva_Afecta_uf,Total_Prima_uf,Afecta_Neta_clp,Iva_clp,Total_Prima_CLP,Poliza,Plan_Tecnico,NOMBRE_ARCHIVO,MES_RECAUDACION
0,None,13280,79,LEASING URBANO,0026869599-0,WILBERT LORVANDAL .,240,5.368199999999999,0,5.3682,0.1503,0.0286,0.1789,5935,1129,7064,8890458,8554,202509 Nomina Cesantia.xlsx,202509
1,None,13278,84,LEASING URBANO,0016663149-1,CARLOS ANDRES VARGAS GONZALEZ,240,4.5151,0,4.5151,0.1264,0.024,0.1504,4991,948,5939,8890458,8554,202509 Nomina Cesantia.xlsx,202509
2,None,13276,80,LEASING URBANO,0025644727-4,JOSE RAFAEL PARRA MEDINA,240,6.991600000000001,0,6.9916,0.1958,0.0372,0.233,7731,1469,9200,8890458,8554,202509 Nomina Cesantia.xlsx,202509
3,None,13275,73,LEASING URBANO,0019574490-4,LAURA ESTER IBAÑEZ FUENTEALBA,240,4.017,0,4.017,0.1125,0.0214,0.1339,4442,845,5287,8890458,8554,202509 Nomina Cesantia.xlsx,202509
4,None,13273,77,LEASING URBANO,0013865957-7,RUBEN EDUARDO LUCERO DIAZ,240,13.524199999999999,0,13.5242,0.3787,0.072,0.4507,14953,2843,17796,8890458,8554,202509 Nomina Cesantia.xlsx,202509


🧩 Bloque 9 – Lógica inteligente de creación condicional

Este bloque:
- detecta si NOMBRE_ARCHIVO o MES_RECAUDACION vienen vacíos o no existen,
- los crea automáticamente solo si faltan,
- respeta los valores cuando vienen desde el archivo.

Evita duplicados y mantiene integridad del dataset.

In [54]:
def _serie_vacia_o_nula(s: pd.Series) -> bool:
    """
    True si TODA la serie está vacía / nula / 'nan' / 'none' / espacios.
    """
    if s is None:
        return True

    # Primero miramos nulos reales (None/NaN)
    if s.isna().all():
        return True

    # Para los que no son NaN, revisamos contenido real
    s_str = s.astype(str).str.strip().str.lower()

    # Consideramos vacíos: "", "nan", "none"
    mask_contenido_real = ~s_str.isin(["", "nan", "none"])

    return not mask_contenido_real.any()


if "NOMBRE_ARCHIVO" not in df_can.columns or _serie_vacia_o_nula(df_can["NOMBRE_ARCHIVO"]):
    nombre_canonico = canonicalizar_planes(archivo_origen.name)
    print("Nombre original :", archivo_origen.name)
    print("Nombre canónico :", nombre_canonico)
    df_can["NOMBRE_ARCHIVO"] = nombre_canonico
else:
    print("✅ NOMBRE_ARCHIVO viene desde el archivo, no se modifica.")


if "MES_RECAUDACION" not in df_can.columns or _serie_vacia_o_nula(df_can["MES_RECAUDACION"]):
    df_can["MES_RECAUDACION"] = extraer_yyyymm_de_nombre_archivo(df_can)

    if df_can["MES_RECAUDACION"].isna().any():
        print("⚠️ Algunos valores de MES_RECAUDACION no pudieron ser extraídos desde NOMBRE_ARCHIVO.")
else:
    print("✅ MES_RECAUDACION viene desde el archivo, no se modifica.")

print(df_can[["NOMBRE_ARCHIVO", "MES_RECAUDACION"]].head(3))


✅ NOMBRE_ARCHIVO viene desde el archivo, no se modifica.
✅ MES_RECAUDACION viene desde el archivo, no se modifica.
                NOMBRE_ARCHIVO MES_RECAUDACION
0  202509 Nomina Cesantia.xlsx          202509
1  202509 Nomina Cesantia.xlsx          202509
2  202509 Nomina Cesantia.xlsx          202509


🔎 Bloque 10 – Normalización avanzada del RUT

Acciones:
- limpia valores,
- separa rut y dígito verificador,
- elimina caracteres no válidos,
- transforma a tipos numéricos seguros (Int64).

Deja el RUT preparado para validación o carga en SQL.

In [55]:
def normaliza_rut_nueva(df, col_rut, col_dv=None):
    """
    Crea nuevas columnas RUT2 y DV2 con el RUT normalizado.
    No modifica las columnas originales.
    """
     # Copia limpia de la columna RUT original
    s = df[col_rut].astype(str).str.upper().str.strip()
    s = s.str.replace(r"[^0-9K]", "", regex=True)  # elimina puntos, guiones, etc.

    df["rut2"] = pd.NA
    df["dv2"] = pd.NA

    if col_dv is None:
        df["dv2"] = s.str[-1]
        df["rut2"] = s.str[:-1]
    else:
        s_dv = df[col_dv].astype(str).str.upper().str.strip()
        s_dv = s_dv.str.replace(r"[^0-9K]", "", regex=True)
        df["rut2"] = s
        df["dv2"] = s_dv

    df["rut2"] = pd.to_numeric(df["rut2"], errors="coerce").astype("Int64")

    print(f"✅ Nuevas columnas 'rut2' y 'dv2' creadas. Total filas: {len(df)}")


normaliza_rut_nueva(df_can, "Rut")

print(df_can[["Rut", "rut2", "dv2"]].head())

✅ Nuevas columnas 'rut2' y 'dv2' creadas. Total filas: 5593
            Rut      rut2 dv2
0  0026869599-0  26869599   0
1  0016663149-1  16663149   1
2  0025644727-4  25644727   4
3  0019574490-4  19574490   4
4  0013865957-7  13865957   7


🧮 Bloque 11 – Tipificación y preparación previa a SQL

Este bloque garantiza:
- conversión segura de enteros, bigints y decimales,
- fechas a datetime,
- textos limpios y recortados,
- revisión de nulos críticos,
- generación de df_sql con el orden exacto requerido por SQL Server.

Deja el DataFrame alineado perfectamente al modelo SQL.

In [56]:
INT_COLS    = ["Guia", "Operacion", "Svs", "Plazo", "Poliza", "Plan_tecnico"]
BIGINT_COLS = ["rut2"]
FLOAT_COLS  = ["Cuota", "Subsidio", "Cuota_Neta", "Prima_Afecta_uf", "Iva_Afecta_uf", "Total_Prima_uf", "Afecta_Neta_clp", "Iva_clp", "Total_Prima_CLP"]
STR_COLS    = ["Dueno", "Rut", "Cliente", "NOMBRE_ARCHIVO", "MES_RECAUDACION", "dv2"]
DATE_COLS   = []


def _to_num_series(s: pd.Series) -> pd.Series:
    s2 = s.astype(str).str.strip().replace({
        "": np.nan,
        "None": np.nan,
        "none": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
    })
    return pd.to_numeric(s2, errors="coerce")


def convert_to_datetime(df: pd.DataFrame, date_columns: list) -> pd.DataFrame:
    """
    Convierte las columnas de fechas de formato object a formato datetime (YYYY-MM-DD).
    Si la conversión falla, reemplaza con NaT (Not a Time).
    """
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
            # Reemplaza cualquier valor inválido con NaT (not a time) si no se puede convertir
            # Esto asegura que las fechas no válidas sean manejadas adecuadamente.
    return df

df_can = convert_to_datetime(df_can, DATE_COLS)


# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        s = _to_num_series(df_can[c])

        frac_mask = (s.notna()) & (s % 1 != 0)
        if frac_mask.any():
            print(f"⚠️ Columna '{c}' (INT) tiene {frac_mask.sum()} valores con decimales. Se redondearán antes de convertir a Int64.")
            # Aproximar al entero más cercano
            s = s.round(0)

        df_can[c] = s.astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")



if "Dueno" in df_can.columns:
    df_can["Dueno"] = df_can["Dueno"].astype("string").str.strip().str.slice(0, 100)

if "Rut" in df_can.columns:
    df_can["Rut"] = df_can["Rut"].astype("string").str.strip().str.slice(0, 24)

if "Cliente" in df_can.columns:
    df_can["Cliente"] = df_can["Cliente"].astype("string").str.strip().str.slice(0, 100)

if "NOMBRE_ARCHIVO" in df_can.columns:
    df_can["NOMBRE_ARCHIVO"] = df_can["NOMBRE_ARCHIVO"].astype("string").str.strip().str.slice(0, 200)

if "MES_RECAUDACION" in df_can.columns:
    df_can["MES_RECAUDACION"] = df_can["MES_RECAUDACION"].astype("string").str.strip().str.slice(0, 12)

if "dv2" in df_can.columns:
    df_can["dv2"] = df_can["dv2"].astype("string").str.strip().str.slice(0, 2)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)


criticas = ["Poliza", "Plan_Tecnico", "NOMBRE_ARCHIVO", "MES_RECAUDACION", "rut2"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "Guia", "Operacion", "Svs", "Dueno", "Rut", "Cliente", "Plazo", "Cuota",
    "Subsidio", "Cuota_Neta", "Prima_Afecta_uf", "Iva_Afecta_uf", "Total_Prima_uf", "Afecta_Neta_clp",
    "Iva_clp", "Total_Prima_CLP", "Poliza", "Plan_tecnico", "NOMBRE_ARCHIVO", "MES_RECAUDACION", "rut2", "dv2",
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_25712\3132993860.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s2 = s.astype(str).str.strip().replace({


✅ dtypes DESPUÉS:
 Guia                        Int64
Operacion                   Int64
SVS                        object
Dueno              string[python]
Rut                string[python]
Cliente            string[python]
Plazo                       Int64
Cuota                     float64
Subsidio                  float64
Cuota_Neta                float64
Prima_Afecta_uf           float64
Iva_Afecta_uf             float64
Total_Prima_uf            float64
Afecta_Neta_clp           float64
Iva_clp                   float64
Total_Prima_CLP           float64
Poliza                      Int64
Plan_Tecnico               object
NOMBRE_ARCHIVO     string[python]
MES_RECAUDACION    string[python]
rut2                        Int64
dv2                string[python]
dtype: object

🔎 Nulos en columnas críticas:
 - Poliza: 0 nulos
 - Plan_Tecnico: 0 nulos
 - NOMBRE_ARCHIVO: 0 nulos
 - MES_RECAUDACION: 0 nulos
 - rut2: 0 nulos

📦 df_sql listo con columnas: ['Guia', 'Operacion', 'Dueno', 'Rut', 'Cli

🧾 Bloque 12 – Validación de unicidad y control de carga

Aquí se:
- valida que exista NOMBRE_ARCHIVO,
- detectan archivos únicos,
- cuentan filas por nombre,
- revisa coherencia del lote previo a insertar.

Permite un control completo antes de manipular la tabla.

In [57]:
assert "NOMBRE_ARCHIVO" in df_sql.columns, "Falta la columna 'NOMBRE_ARCHIVO' en df_sql."

df_sql["NOMBRE_ARCHIVO"] = (
    df_sql["NOMBRE_ARCHIVO"]
    .astype("string")
    .str.strip()
)


vals = [
    v for v in df_sql["NOMBRE_ARCHIVO"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'NOMBRE_ARCHIVO'.")

counts = (
    df_sql["NOMBRE_ARCHIVO"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} NOMBRE_ARCHIVO distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 3 NOMBRE_ARCHIVO distintos en el df_sql:
   - 202510 Nomina Cesantia.xlsx: 1867 filas
   - 202509 Nomina Cesantia.xlsx: 1865 filas
   - 202511 Nomina Cesantia.xlsx: 1861 filas


🔄 Bloque 11 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [58]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    for nombre_archivo in vals:  
        df_sub = df_sql[df_sql["NOMBRE_ARCHIVO"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para NOMBRE_ARCHIVO = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para NOMBRE_ARCHIVO = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para NOMBRE_ARCHIVO = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,       
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para NOMBRE_ARCHIVO = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por NOMBRE_ARCHIVO:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

🆕 No hay data previa para NOMBRE_ARCHIVO = '202509 Nomina Cesantia.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 1865 filas para NOMBRE_ARCHIVO = '202509 Nomina Cesantia.xlsx'.
🆕 No hay data previa para NOMBRE_ARCHIVO = '202510 Nomina Cesantia.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 1867 filas para NOMBRE_ARCHIVO = '202510 Nomina Cesantia.xlsx'.
🆕 No hay data previa para NOMBRE_ARCHIVO = '202511 Nomina Cesantia.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 1861 filas para NOMBRE_ARCHIVO = '202511 Nomina Cesantia.xlsx'.

📊 Resumen de carga por NOMBRE_ARCHIVO:
   - 202509 Nomina Cesantia.xlsx: 1865 filas cargadas (archivo nuevo).
   - 202510 Nomina Cesantia.xlsx: 1867 filas cargadas (archivo nuevo).
   - 202511 Nomina Cesantia.xlsx: 1861 filas cargadas (archivo nuevo).


🗑️ Bloque 14 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [ ]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")